# Phase 4b: LLM-only baselines + significance tests + error analysis

**Settings:** Accelerator **GPU T4 x2** (the LLM uses both GPUs) and **Internet on**.
**Attach (Add Input → Your Work):** `phase3-ablation` (required) and `phase3a-llm` (explanations for the error analysis).

1. The LLM (Qwen2.5-7B, not fine-tuned) labels the 713 test replies directly: zero-shot, then few-shot (2 training examples per class).
2. Paired significance tests for every ablation variant vs base, plus the LLM baselines vs the fine-tuned +LLM models.
3. Error analysis: which test replies the explanation fixes or breaks, with examples.

The report is written to `/kaggle/working/outputs/significance_weighted_f1.md`.

In [ ]:
!nvidia-smi -L
import transformers, torch
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "NO GPU! Settings -> Accelerator -> GPU T4 x2, then run again. (On CPU every run takes hours.)"


In [ ]:
# Clone (or update) the project code and the dataset.
import os, subprocess, glob
REPO = "akashnegetive/stance-aware-llm-explanation-augmented-hate-speech-detection-for-replies-in-conversation"
CODE_DIR = "/kaggle/working/code"
if os.path.isdir(CODE_DIR):
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", CODE_DIR], check=True)
%cd {CODE_DIR}
if not os.path.isdir("data/counter_context"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/xinchenyu/counter_context", "data/counter_context"], check=True)

In [ ]:
import shlex
# Kaggle mounts notebook outputs at varying depths, so search all levels.
prev = sorted(glob.glob("/kaggle/input/**/outputs", recursive=True))
llm_files = sorted(glob.glob("/kaggle/input/**/llm/llm_outputs.jsonl", recursive=True))
assert prev, "Attach the phase3-ablation notebook as an input first"
LLM_FILE = llm_files[0] if llm_files else ""
DIRS = " ".join(shlex.quote(d) for d in prev + ["/kaggle/working/outputs"])
print("Ablation results from:", prev)
print("LLM file:", LLM_FILE or "not attached (error analysis will show no explanations)")

In [ ]:
# Smoke test (~5-10 min, mostly the model download): 20 test pairs, zero-shot. Most answers should parse.
!python llm_classify.py --shots 0 --limit 20 --output_dir /kaggle/working/smoke
!head -n 5 /kaggle/working/smoke/llm_Qwen2.5-7B-Instruct_0shot_lim20/raw.jsonl

In [ ]:
# LLM-only baselines on all 713 test pairs.
!python llm_classify.py --shots 0 --output_dir /kaggle/working/outputs
!python llm_classify.py --shots 2 --output_dir /kaggle/working/outputs

In [ ]:
# Significance tests + error analysis (CPU, ~1-2 min).
!python significance.py --results_dirs {DIRS} --llm_file "{LLM_FILE}" --out_dir /kaggle/working/outputs